In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [4]:
# --- 0. INSTALAR LIBRERÍA ---
!pip install ultralytics -q

import os
import shutil
import yaml
from pathlib import Path
from ultralytics import YOLO

# --- 1. LOCALIZAR EL DATASET (YA DESCOMPRIMIDO POR KAGGLE) ---
print("🕵️ Buscando las carpetas de imágenes en /kaggle/input...")

base_dataset_path = None
# Buscamos directamente la carpeta 'train' dentro de 'images'
for path in Path('/kaggle/input').rglob('train'):
    if path.is_dir() and path.parent.name == 'images':
        base_dataset_path = path.parent.parent # Subimos hasta la raíz del dataset
        break

if not base_dataset_path:
    raise Exception("❌ ERROR: No encuentro las carpetas. Revisa en el panel derecho (Input) que el dataset esté subido.")

print(f"✅ ¡Dataset localizado en: {base_dataset_path}!")

# --- 2. CREAR EL DATA.YAML ---
working_dir = Path('/kaggle/working')
yaml_path = working_dir / 'data.yaml'

yaml_content = {
    'train': str(base_dataset_path / 'images' / 'train'),
    'val': str(base_dataset_path / 'images' / 'val'),
    'test': str(base_dataset_path / 'images' / 'test'),
    'nc': 4,
    'names': ['Alligator Crack', 'Longitudinal Crack', 'Pothole', 'Transverse Crack']
}

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)
    
print("📝 Archivo data.yaml creado con las rutas correctas.")

# --- 3. ENTRENAR YOLO11 MEDIUM ---
print("\n🚀 Iniciando entrenamiento con YOLO11 Medium...")
model = YOLO('yolo11m.pt')

results = model.train(
    data=str(yaml_path),
    epochs=100,       # 100 épocas máximo
    patience=20,      # Se detiene solo si no mejora en 20 épocas
    imgsz=640,
    batch=16,         # Baja esto a 8 si te da error de memoria "CUDA out of memory"
    device=0,
    project='/kaggle/working/runs',
    name='yolo11m_optimizado'
)

# --- 4. EMPAQUETAR RESULTADOS ---
print("\n📦 Guardando y empaquetando todos los resultados...")
shutil.make_archive('/kaggle/working/resultados_finales', 'zip', '/kaggle/working/runs')
print("✅ ¡Proceso completado! El archivo 'resultados_finales.zip' está listo.")

🕵️ Buscando las carpetas de imágenes en /kaggle/input...
✅ ¡Dataset localizado en: /kaggle/input/datasets/gemita284/dataset-limpio!
📝 Archivo data.yaml creado con las rutas correctas.

🚀 Iniciando entrenamiento con YOLO11 Medium...
Ultralytics 8.4.95 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width